In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [3]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

12.25

In [5]:
def get_values(symbol, size, smaa=21, t=1):
    d = {5:mt5.TIMEFRAME_M5, 15:mt5.TIMEFRAME_M15,1:mt5.TIMEFRAME_H1, 2:mt5.TIMEFRAME_H2, 3:mt5.TIMEFRAME_H3,4:mt5.TIMEFRAME_H4,12:mt5.TIMEFRAME_H12, 'd':mt5.TIMEFRAME_D1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    v = niss_Fast(rates_frame['close'])
    rates_frame['nissFast'] = [0]*(len(rates_frame['close']) - len(v)) + v
    v = niss_Slow(rates_frame['close'])
    rates_frame['nissSlow'] = [0]*(len(rates_frame['close']) - len(v)) + v
    
    rates_frame['nissOscRaw'] = [0]*69 + list(map(lambda x,y : ((x - y)/y)*100, rates_frame['nissFast'][69:], rates_frame['nissSlow'][69:]))

    rates_frame['nissOsc'] = rates_frame['nissOscRaw'].rolling(window=1).mean()
    
    v = ema(rates_frame['nissOscRaw'], 24)
    rates_frame['nissSignal'] = [0]*(len(rates_frame['close']) - len(v)) + v
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame['ema'] = rates_frame['close'].ewm(span=50, adjust=False).mean()


    return rates_frame

In [6]:
def ema(s, n):
    ema = []
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)

    return ema

In [7]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [8]:
def niss_Fast(src):
    emaF1 = ema(src, 3)
    emaF2 = ema(src, 5)
    emaF3 = ema(src, 7)
    emaF4 = ema(src, 9)
    emaF5 = ema(src, 11)
    emaF6 = ema(src, 13)
    emaF7 = ema(src, 15)
    emaF8 = ema(src, 17)
    emaF9 = ema(src, 19)
    emaF10 = ema(src, 21)
    emaF11 = ema(src, 23)
    return list(map(lambda x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11:(x1+x2+x3+x4+x5+x6+x7+x8+x9+x10+x11)/11, \
             emaF1[len(emaF1)-len(emaF11):], \
             emaF2[len(emaF2)-len(emaF11):],  emaF3[len(emaF3)-len(emaF11):], emaF4[len(emaF4)-len(emaF11):], \
            emaF5[len(emaF5)-len(emaF11):], emaF6[len(emaF6)-len(emaF11):], emaF7[len(emaF7)-len(emaF11):], \
            emaF8[len(emaF8)-len(emaF11):], emaF9[len(emaF9)-len(emaF11):], emaF10[len(emaF10)-len(emaF11):], \
            emaF11))
    
def niss_Slow(src):
    emaS1 = ema(src, 25)
    emaS2 = ema(src, 28)
    emaS3 = ema(src, 31)
    emaS4 = ema(src, 34)
    emaS5 = ema(src, 37)
    emaS6 = ema(src, 40)
    emaS7 = ema(src, 43)
    emaS8 = ema(src, 46)
    emaS9 = ema(src, 49)
    emaS10 = ema(src, 52)
    emaS11 = ema(src, 55)
    emaS12 = ema(src, 58)
    emaS13 = ema(src, 61)
    emaS14 = ema(src, 64)
    emaS15 = ema(src, 67)
    emaS16 = ema(src, 70)
    return list(map(lambda x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, x13, x14, x15, x16: \
                    (x1+x2+x3+x4+x5+x6+x7+x8+x9+x10+x11+x12+x13+x14+x15+x16)/16, \
         emaS1[len(emaS1)-len(emaS16):], \
         emaS2[len(emaS2)-len(emaS16):],  emaS3[len(emaS3)-len(emaS16):], emaS4[len(emaS4)-len(emaS16):], \
        emaS5[len(emaS5)-len(emaS16):], emaS6[len(emaS6)-len(emaS16):], emaS7[len(emaS7)-len(emaS16):], \
        emaS8[len(emaS8)-len(emaS16):], emaS9[len(emaS9)-len(emaS16):], emaS10[len(emaS10)-len(emaS16):], \
        emaS11[len(emaS11)-len(emaS16):], emaS12[len(emaS12)-len(emaS16):], emaS13[len(emaS13)-len(emaS16):], \
                   emaS14[len(emaS14)-len(emaS16):], emaS15[len(emaS15)-len(emaS16):], emaS16))

In [161]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "GBPUSD"
a = get_values(symbol, 2000, 21, 4)
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


def close(buy_price, i,a,utc_from):
    check = 0
    t = datetime.now(tz=timezone)
    utc_to = datetime(t.year, t.month, t.day, hour=t.hour, minute=t.minute)
    rates = mt5.copy_rates_range("GBPUSD", mt5.TIMEFRAME_M1, utc_from, utc_to)
#     rates = mt5.copy_rates_from("GBPUSD", mt5.TIMEFRAME_M1, utc_from, 10000)
    b = pd.DataFrame(rates)
    b['time'] = pd.to_datetime(b['time'], unit='s')
    old_pp = 0.0
    minn = -20.0
    for j in range(1, len(b)):
        sell_price = b.iloc[j].close
        pp = price_action(symbol, 0.05, buy_price, sell_price,mt5.ORDER_TYPE_SELL) - 1.60
        
        if pp < old_pp:
            old_pp = pp
#         loss = -10.0
        pro = 5.0
#         if pp < loss:
#             profit.append(loss)
#             break
        if pp > pro:
            check = 1
            print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
            profit.append(pp)
            max_loss.append(old_pp)
            break
            
        if pp <= minn:
            check = 1
            print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
            profit.append(pp)
            max_loss.append(old_pp)
            break
    if check==0:
        print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
        index.append(1)
        profit.append(pp)
        max_loss.append(old_pp)

t1 = time.time()
for i in range(1,len(a)):
    
    if a.iloc[i].close < a.iloc[i].sma:
        if a.iloc[i].nissOsc < a.iloc[i].nissSignal: #and check == 0:# and  a.iloc[i].close < a.iloc[i].sma:
            buy_price = a.iloc[i].close #SELL Action
            print(f"buy ---{a.iloc[i].close}--{a.iloc[i].name}")
            print("="*30)
    #         utc_from = datetime(2023, 10, 30,hour=6, minute=30, tzinfo=timezone)
#             close(buy_price, i, a, a.iloc[i].name)
            t1 = threading.Thread(target=close, args=(buy_price, i,a,a.iloc[i].name)).start()
            print("*"*30)
    #         break
    
# print(time.time() - t1)
        


buy ---1.19237--2023-01-03 08:00:00
******************************
buy ---1.19931--2023-01-03 12:00:00
******************************
buy ---1.19642--2023-01-03 16:00:00
******************************
buy ---1.19655--2023-01-03 20:00:00
******************************
buy ---1.1993800000000001--2023-01-04 00:00:00
******************************
buy ---1.19846--2023-01-04 04:00:00
******************************
buy ---1.19199--2023-01-05 12:00:00
******************************
buy ---1.19073--2023-01-05 16:00:00
******************************
buy ---1.19128--2023-01-05 20:00:00
******************************
buy ---1.19185--2023-01-06 00:00:00
******************************
buy ---1.19039--2023-01-06 04:00:00
******************************
buy ---1.18582--2023-01-06 08:00:00
******************************
buy ---1.19262--2023-01-06 12:00:00
******************************
buy ---1.2190699999999999--2023-01-16 16:00:00
******************************
buy ---1.21917--2023-01-16 20:00:00
****

Exception in thread Thread-27549:
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.7_qbz5n2kfra8p0\LocalCache\local-packages\Python37\site-packages\pandas\core\indexes\base.py", line 3361, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas\_libs\index.pyx", line 76, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\index.pyx", line 108, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\hashtable_class_helper.pxi", line 5198, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\_libs\hashtable_class_helper.pxi", line 5206, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'time'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.7_3.7.2544.0_x64__qbz5n2kfra8p0\lib\threading.py", line 926, in _bootstrap_inner
    self.run()
  Fi

******************************
buy ---1.20135--2023-02-23 20:00:00
******************************
buy ---1.20161--2023-02-24 00:00:00
******************************
buy ---1.20229--2023-02-24 04:00:00
******************************
buy ---1.20237--2023-02-24 08:00:00
******************************
buy ---1.19647--2023-02-24 12:00:00
******************************
buy ---1.19457--2023-02-24 16:00:00
******************************
buy ---1.19392--2023-02-24 20:00:00
******************************
buy ---1.19625--2023-02-27 00:00:00
******************************
buy ---1.19381--2023-02-27 04:00:00
******************************
buy ---1.1973799999999999--2023-02-27 08:00:00
******************************
buy ---1.19882--2023-03-02 04:00:00
******************************
buy ---1.19732--2023-03-02 08:00:00
******************************
buy ---1.19288--2023-03-02 12:00:00
******************************
buy ---1.19335--2023-03-02 16:00:00
******************************
buy ---1.19478--2023

Exception in thread Thread-27628:
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.7_qbz5n2kfra8p0\LocalCache\local-packages\Python37\site-packages\pandas\core\indexes\base.py", line 3361, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas\_libs\index.pyx", line 76, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\index.pyx", line 108, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\hashtable_class_helper.pxi", line 5198, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\_libs\hashtable_class_helper.pxi", line 5206, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'time'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.7_3.7.2544.0_x64__qbz5n2kfra8p0\lib\threading.py", line 926, in _bootstrap_inner
    self.run()
  Fi

******************************
buy ---1.24098--2023-04-17 12:00:00
******************************
buy ---1.23561--2023-04-17 16:00:00
******************************
buy ---1.23769--2023-04-17 20:00:00
******************************
buy ---1.23729--2023-04-18 00:00:00
******************************
buy ---1.2383899999999999--2023-04-18 04:00:00
******************************
buy ---1.24381--2023-04-18 08:00:00
******************************
buy ---1.24291--2023-04-18 12:00:00
******************************
buy ---1.24286--2023-04-18 16:00:00
******************************
buy ---1.24254--2023-04-18 20:00:00
******************************
buy ---1.24311--2023-04-19 00:00:00
******************************
buy ---1.24186--2023-04-19 04:00:00
******************************
buy ---1.24--2023-04-21 08:00:00
******************************
buy ---1.24135--2023-04-21 12:00:00
******************************
buy ---1.24302--2023-04-21 16:00:00
******************************
buy ---1.23994--2023-04

...

******************************
buy ---1.26472--2023-12-18 20:00:00
******************************
buy ---1.26528--2023-12-19 00:00:00
******************************
buy ---1.26581--2023-12-19 04:00:00
******************************
buy ---1.26611--2023-12-20 12:00:00
******************************
buy ---1.26701--2023-12-20 16:00:00
******************************
buy ---1.26376--2023-12-20 20:00:00
******************************
buy ---1.26422--2023-12-21 00:00:00
******************************
buy ---1.26494--2023-12-21 04:00:00
******************************
buy ---1.2631299999999999--2023-12-21 08:00:00
******************************
buy ---1.26642--2023-12-21 16:00:00
******************************
buy ---1.27367--2023-12-29 00:00:00
******************************
buy ---1.27057--2023-12-29 08:00:00
******************************
buy ---1.2721--2023-12-29 12:00:00
******************************
buy ---1.27301--2023-12-29 20:00:00
******************************
buy ---1.27206--2024-

Exception in thread Thread-28164:
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.7_qbz5n2kfra8p0\LocalCache\local-packages\Python37\site-packages\pandas\core\indexes\base.py", line 3361, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas\_libs\index.pyx", line 76, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\index.pyx", line 108, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\hashtable_class_helper.pxi", line 5198, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\_libs\hashtable_class_helper.pxi", line 5206, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'time'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.7_3.7.2544.0_x64__qbz5n2kfra8p0\lib\threading.py", line 926, in _bootstrap_inner
    self.run()
  Fi

******************************
buy ---1.2699--2024-01-26 20:00:00
******************************
buy ---1.27004--2024-01-29 00:00:00
******************************
buy ---1.27019--2024-01-29 04:00:00
******************************
buy ---1.27034--2024-01-29 08:00:00
******************************
buy ---1.2693699999999999--2024-01-29 12:00:00
******************************
buy ---1.26834--2024-01-29 16:00:00
pp: 12.450000000000001---1.26569--2024-01-23 20:01:00
******************************
buy ---1.27074--2024-01-29 20:00:00
******************************
buy ---1.27104--2024-01-30 00:00:00
******************************
buy ---1.2705600000000001--2024-01-30 04:00:00
pp: 6.1---1.26906--2024-01-25 20:01:00
pp: 8.85---1.26701--2024-01-24 00:01:00
******************************
buy ---1.26811--2024-01-30 08:00:00
pp: 5.1---1.27004--2024-01-26 08:07:00
******************************
buy ---1.26752--2024-01-30 12:00:00
******************************
buy ---1.26807--2024-01-30 16:00:00
***

pp: 5.35---1.25477--2024-02-15 09:39:00
pp: 5.1---1.26457--2024-02-28 09:36:00
******************************
buy ---1.26303--2024-03-01 08:00:00
******************************
buy ---1.26302--2024-03-01 12:00:00
pp: -20.3---1.25888--2024-02-15 16:34:00
pp: 8.6---1.25648--2024-02-15 16:01:00
pp: 5.050000000000001---1.26524--2024-02-29 09:57:00
pp: 5.949999999999999---1.26368--2024-02-29 15:00:00
******************************
buy ---1.26509--2024-03-01 20:00:00
******************************
buy ---1.27901--2024-03-12 08:00:00
******************************
buy ---1.27604--2024-03-12 12:00:00
pp: -20.25---1.26523--2024-02-29 16:03:00
pp: 6.65---1.26391--2024-02-28 16:01:00
******************************
buy ---1.27839--2024-03-12 16:00:00
pp: 5.75---1.26514--2024-02-29 13:36:00
******************************
buy ---1.27888--2024-03-12 20:00:00
pp: 5.25---1.2643900000000001--2024-02-29 14:11:00
******************************
buy ---1.27923--2024-03-13 00:00:00
pp: 8.6---1.26127000000000

******************************
buy ---1.26271--2024-03-26 16:00:00
pp: -20.450000000000003---1.26278--2024-03-25 13:21:00
******************************
buy ---1.26282--2024-03-26 20:00:00
pp: -20.5---1.26325--2024-03-25 13:51:00
******************************
buy ---1.2616--2024-03-27 00:00:00
******************************
buy ---1.26142--2024-03-27 04:00:00
******************************
buy ---1.26261--2024-03-27 08:00:00
******************************pp: 5.4---1.26276--2024-03-26 16:27:00

buy ---1.26177--2024-03-27 12:00:00
******************************
pp: 5.6---1.26276--2024-03-26 16:27:00pp: 5.199999999999999---1.26144--2024-03-27 03:31:00

pp: 5.15---1.2640500000000001--2024-03-26 14:44:00
pp: 5.15---1.26221--2024-03-26 18:23:00
pp: 5.25---1.2610999999999999--2024-03-27 13:57:00
pp: 6.0---1.26306--2024-03-26 16:15:00
pp: 5.25---1.26251--2024-03-26 16:30:00
pp: 5.65---1.26126--2024-03-27 03:39:00


In [162]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

219.24999999999991
Total negative sm -->-487.25000000000006
Total negative -->22
Total positive sm -->706.5000000000002
Total positive -->107
Length 129
pp: 5.449999999999999---1.25999--2024-03-28 11:08:00
pp: 5.6---1.26033--2024-03-28 11:03:00
pp: 5.35---1.26019--2024-03-28 11:07:00


In [163]:
profit.sort()
profit

[-34.15,
 -27.700000000000003,
 -26.25,
 -26.05,
 -22.35,
 -22.0,
 -21.8,
 -21.650000000000002,
 -21.0,
 -20.900000000000002,
 -20.55,
 -20.5,
 -20.450000000000003,
 -20.450000000000003,
 -20.3,
 -20.3,
 -20.25,
 -20.200000000000003,
 -20.150000000000002,
 -20.150000000000002,
 -20.05,
 -20.05,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.050000000000001,
 5.1,
 5.1,
 5.1,
 5.15,
 5.15,
 5.15,
 5.15,
 5.15,
 5.199999999999999,
 5.199999999999999,
 5.199999999999999,
 5.199999999999999,
 5.199999999999999,
 5.25,
 5.25,
 5.25,
 5.25,
 5.25,
 5.25,
 5.25,
 5.300000000000001,
 5.300000000000001,
 5.300000000000001,
 5.300000000000001,
 5.300000000000001,
 5.300000000000001,
 5.35,
 5.35,
 5.35,
 5.35,
 5.4,
 5.4,
 5.4,
 5.449999999999999,
 5.449999999999999,
 5.449999999999999,
 5.449999999999999,
 5.5,
 5.5,
 5.5,
 5.5,
 5.550000000000001,
 5.550000000000001,
 5.6,
 5.6,


In [164]:
max_loss.sort()
max_loss

[-34.15,
 -27.700000000000003,
 -26.25,
 -26.05,
 -22.35,
 -22.0,
 -21.8,
 -21.650000000000002,
 -21.0,
 -20.900000000000002,
 -20.55,
 -20.5,
 -20.450000000000003,
 -20.450000000000003,
 -20.3,
 -20.3,
 -20.25,
 -20.200000000000003,
 -20.150000000000002,
 -20.150000000000002,
 -20.05,
 -20.05,
 -19.6,
 -18.200000000000003,
 -17.400000000000002,
 -16.5,
 -16.35,
 -15.45,
 -15.299999999999999,
 -15.299999999999999,
 -14.9,
 -14.75,
 -14.5,
 -14.45,
 -14.35,
 -14.25,
 -14.15,
 -13.85,
 -13.6,
 -13.0,
 -12.85,
 -12.799999999999999,
 -12.65,
 -12.2,
 -12.1,
 -11.9,
 -11.85,
 -11.549999999999999,
 -11.25,
 -11.2,
 -11.2,
 -11.1,
 -10.85,
 -10.65,
 -10.5,
 -10.4,
 -10.1,
 -9.75,
 -9.7,
 -9.55,
 -9.35,
 -9.15,
 -9.1,
 -8.85,
 -8.8,
 -8.8,
 -8.45,
 -8.35,
 -8.25,
 -8.1,
 -8.1,
 -8.1,
 -8.05,
 -7.65,
 -7.550000000000001,
 -7.5,
 -7.449999999999999,
 -7.35,
 -7.35,
 -7.15,
 -7.1,
 -7.050000000000001,
 -6.9,
 -6.800000000000001,
 -6.699999999999999,
 -6.65,
 -6.550000000000001,
 -5.75,
 -5.35,
 -

In [165]:
sum(max_loss)

-1259.3499999999997

In [155]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "GBPUSD"
a = get_values(symbol, 1000, 21, 4)
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


def close(buy_price, i,a,utc_from):
    check = 0
    t = datetime.now(tz=timezone)
    utc_to = datetime(t.year, t.month, t.day, hour=t.hour, minute=t.minute)
    rates = mt5.copy_rates_range("GBPUSD", mt5.TIMEFRAME_M15, utc_from, utc_to)
#     rates = mt5.copy_rates_from("GBPUSD", mt5.TIMEFRAME_M1, utc_from, 10000)
    b = pd.DataFrame(rates)
    b['time'] = pd.to_datetime(b['time'], unit='s')
    old_pp = 0.0
    for j in range(1, len(b)):
        sell_price = b.iloc[j].low
        pp = price_action(symbol, 1.0, buy_price, sell_price,mt5.ORDER_TYPE_SELL) - 10.60
        
        if pp < old_pp:
            old_pp = pp
        loss = -200.0
        pro = 5.0
#         if pp < loss:
#             profit.append(loss)
#             break
        if pp > pro:
            check = 1
            print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
            profit.append(pp)
            max_loss.append(old_pp)
            break
        if pp <loss:
            check = 1
            print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
            profit.append(pp)
            max_loss.append(old_pp)
            break
    if check==0:
        print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
        index.append(1)
        profit.append(pp)
        max_loss.append(old_pp)

t1 = time.time()
for i in range(1,len(a)):
    
    if a.iloc[i].close < a.iloc[i].sma:
        if a.iloc[i].nissOsc < a.iloc[i].nissSignal: #and check == 0:# and  a.iloc[i].close < a.iloc[i].sma:
            buy_price = a.iloc[i].close #SELL Action
            print(f"buy ---{a.iloc[i].close}--{a.iloc[i].name}")
            print("="*30)
    #         utc_from = datetime(2023, 10, 30,hour=6, minute=30, tzinfo=timezone)
#             close(buy_price, i, a, a.iloc[i].name)
            t1 = threading.Thread(target=close, args=(buy_price, i,a,a.iloc[i].name)).start()
#             print("*"*30)
    #         break
    
# print(time.time() - t1)
        


buy ---1.27189--2023-08-24 00:00:00
buy ---1.27199--2023-08-24 04:00:00
buy ---1.26985--2023-08-24 08:00:00
buy ---1.26358--2023-08-24 12:00:00
buy ---1.2623199999999999--2023-08-24 16:00:00
buy ---1.2600500000000001--2023-08-24 20:00:00
buy ---1.25896--2023-08-25 00:00:00
buy ---1.25647--2023-08-25 04:00:00
buy ---1.2586300000000001--2023-08-25 08:00:00
buy ---1.2619799999999999--2023-08-25 12:00:00
pp: 31.4---1.2714699999999999--2023-08-24 00:15:00
buy ---1.25818--2023-08-25 16:00:00
pp: 118.4---1.2707--2023-08-24 04:15:00
buy ---1.2576800000000001--2023-08-25 20:00:00
buy ---1.25935--2023-08-28 00:00:00
buy ---1.25996--2023-08-28 04:00:00
pp: -238.6---1.2646--2023-08-24 17:45:00
pp: -231.6---1.25868--2023-08-25 04:15:00
buy ---1.25759--2023-08-28 08:00:00
pp: -467.6---1.2681499999999999--2023-08-24 12:15:00
pp: -228.6---1.27203--2023-08-24 08:15:00
buy ---1.25939--2023-08-28 12:00:00
pp: 27.4---1.25858--2023-08-25 02:00:00
pp: 52.4---1.25942--2023-08-24 22:30:00
buy ---1.2595--2023-

buy ---1.22113--2023-09-25 16:00:00pp: 224.4---1.22333--2023-09-22 12:15:00

pp: 9.4---1.22396--2023-09-25 04:30:00
pp: 29.4---1.22821--2023-09-22 03:15:00
pp: 23.4---1.22305--2023-09-22 21:30:00
pp: 34.4---1.22123--2023-09-25 12:15:00
pp: -316.6---1.22835--2023-09-22 08:15:00
buy ---1.22106--2023-09-25 20:00:00
pp: 57.4---1.22416--2023-09-25 00:15:00
pp: -211.6---1.22715--2023-09-22 17:00:00
buy ---1.22054--2023-09-26 00:00:00
pp: -206.6---1.22398--2023-09-25 08:15:00
buy ---1.2205300000000001--2023-09-26 04:00:00
buy ---1.21847--2023-09-26 08:00:00
pp: 59.4---1.22043--2023-09-25 16:15:00
buy ---1.21908--2023-09-26 12:00:00
pp: 25.4---1.22018--2023-09-26 00:15:00
buy ---1.21708--2023-09-26 16:00:00
buy ---1.21577--2023-09-26 20:00:00
pp: 7.4---1.22088--2023-09-25 21:30:00
buy ---1.21422--2023-09-27 00:00:00
pp: 5.4---1.22037--2023-09-26 04:30:00
buy ---1.21469--2023-09-27 04:00:00
buy ---1.21495--2023-09-27 08:00:00
pp: 72.4---1.21764--2023-09-26 09:15:00
buy ---1.21444--2023-09-27 12

pp: 84.4---1.26129--2023-11-30 20:45:00
buy ---1.2636--2023-12-05 00:00:00
pp: -518.6---1.2699500000000001--2023-11-30 08:15:00
pp: -292.6---1.26578--2023-12-01 12:15:00
buy ---1.26391--2023-12-05 04:00:00
pp: 26.4---1.26406--2023-12-01 05:30:00
buy ---1.2631299999999999--2023-12-05 08:00:00
pp: 365.4---1.26152--2023-12-01 00:15:00
pp: 112.4---1.26237--2023-12-05 00:15:00
buy ---1.26363--2023-12-05 12:00:00
pp: 28.4---1.2652999999999999--2023-12-04 12:15:00
pp: 98.4---1.26115--2023-12-04 16:15:00
buy ---1.25952--2023-12-05 16:00:00
buy ---1.25928--2023-12-05 20:00:00
pp: 50.4---1.2633--2023-12-05 04:15:00
buy ---1.26025--2023-12-06 00:00:00
pp: 239.4---1.2607--2023-12-04 20:15:00
pp: 24.4---1.26278--2023-12-05 08:30:00
buy ---1.2610700000000001--2023-12-06 04:00:00
buy ---1.26--2023-12-06 08:00:00
pp: -258.6---1.262--2023-12-05 16:15:00
buy ---1.2598799999999999--2023-12-06 12:00:00
pp: 65.4---1.25852--2023-12-05 20:15:00
pp: 160.4---1.25854--2023-12-06 00:15:00
pp: 37.4---1.26315--202

pp: 137.4---1.27128--2024-01-15 12:15:00
buy ---1.26947--2024-01-16 00:00:00
pp: 102.4---1.27179--2024-01-15 16:15:00
buy ---1.26804--2024-01-16 04:00:00
pp: 50.4---1.27207--2024-01-16 00:00:00
buy ---1.2640799999999999--2024-01-16 08:00:00
pp: 12.4---1.2678099999999999--2024-01-16 05:30:00
buy ---1.26424--2024-01-16 12:00:00
pp: -375.6---1.26773--2024-01-16 08:15:00
buy ---1.26248--2024-01-16 16:00:00
pp: 88.4---1.26325--2024-01-16 12:15:00
pp: -306.6---1.27243--2024-01-16 00:15:00
buy ---1.26358--2024-01-16 20:00:00
buy ---1.26355--2024-01-17 00:00:00
pp: -258.6---1.2649599999999999--2024-01-16 16:15:00
buy ---1.26098--2024-01-17 04:00:00
pp: 72.4---1.26275--2024-01-16 20:15:00
buy ---1.26803--2024-01-17 08:00:00
buy ---1.26507--2024-01-17 12:00:00
buy ---1.26783--2024-01-17 16:00:00
buy ---1.26759--2024-01-17 20:00:00
pp: -324.6---1.26821--2024-01-17 12:15:00
pp: 34.4---1.2631000000000001--2024-01-17 00:15:00
pp: 307.4---1.26465--2024-01-17 16:15:00
buy ---1.26742--2024-01-18 08:00:

pp: 48.4---1.26605--2024-02-29 04:15:00
buy ---1.26523--2024-02-29 12:00:00
buy ---1.26172--2024-02-29 16:00:00
buy ---1.26207--2024-02-29 20:00:00
pp: 42.4---1.26612--2024-02-29 08:15:00
buy ---1.26342--2024-03-01 00:00:00
pp: 41.4---1.26155--2024-02-29 20:15:00
pp: -439.6---1.26601--2024-02-29 16:15:00
buy ---1.2625899999999999--2024-03-01 04:00:00
buy ---1.26303--2024-03-01 08:00:00
pp: 35.4---1.26477--2024-02-29 13:30:00
pp: 154.4---1.26177--2024-03-01 00:15:00
pp: 29.4---1.26263--2024-03-01 08:15:00
buy ---1.26302--2024-03-01 12:00:00
pp: 11.4---1.26237--2024-03-01 07:30:00
buy ---1.26509--2024-03-01 20:00:00
pp: 117.4---1.26174--2024-03-01 12:15:00
pp: 40.4---1.26458--2024-03-01 23:45:00
buy ---1.27901--2024-03-12 08:00:00
buy ---1.27604--2024-03-12 12:00:00
buy ---1.27839--2024-03-12 16:00:00
pp: -290.6---1.27884--2024-03-12 12:15:00
pp: -245.6---1.28136--2024-03-12 08:15:00
pp: 287.4---1.27541--2024-03-12 16:15:00
buy ---1.27888--2024-03-12 20:00:00
buy ---1.27923--2024-03-13 0

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 1000, 21, 1)
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


def close(buy_price, i,a,utc_from):
    check = 0
    t = datetime.now(tz=timezone)
    utc_to = datetime(t.year, t.month, t.day, hour=t.hour, minute=t.minute)
    rates = mt5.copy_rates_range("BTCUSD", mt5.TIMEFRAME_H1, utc_from, utc_to)
#     rates = mt5.copy_rates_from("GBPUSD", mt5.TIMEFRAME_M1, utc_from, 10000)
    b = pd.DataFrame(rates)
    b['time'] = pd.to_datetime(b['time'], unit='s')
    old_pp = 0.0
    for j in range(1, len(b)):
        sell_price = b.iloc[j].low
        pp = price_action(symbol, 1.0, buy_price, sell_price,mt5.ORDER_TYPE_SELL) - 10.60
        
        if pp < old_pp:
            old_pp = pp
        loss = -200.0
        pro = 5.0
#         if pp < loss:
#             profit.append(loss)
#             break
        if pp > pro:
            check = 1
            print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
            profit.append(pp)
            max_loss.append(old_pp)
            break
        if pp <loss:
            check = 1
            print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
            profit.append(pp)
            max_loss.append(old_pp)
            break
    if check==0:
        print(f"pp: {pp}---{b.iloc[j].low}--{b.iloc[j].time}")
        index.append(1)
        profit.append(pp)
        max_loss.append(old_pp)

t1 = time.time()
for i in range(1,len(a)):
    
    if a.iloc[i].close < a.iloc[i].sma:
        if a.iloc[i].nissOsc < a.iloc[i].nissSignal: #and check == 0:# and  a.iloc[i].close < a.iloc[i].sma:
            buy_price = a.iloc[i].close #SELL Action
            print(f"buy ---{a.iloc[i].close}--{a.iloc[i].name}")
            print("="*30)
    #         utc_from = datetime(2023, 10, 30,hour=6, minute=30, tzinfo=timezone)
#             close(buy_price, i, a, a.iloc[i].name)
            t1 = threading.Thread(target=close, args=(buy_price, i,a,a.iloc[i].name)).start()
#             print("*"*30)
    #         break
    
# print(time.time() - t1)
        
